In [ ]:
"""
visualization.py

Creates visualizations for U-Net PINN LST predictions.

Outputs:

    1. Reference LST map
    2. Predicted LST map
    3. Prediction error map
    4. Training and validation loss curves
"""

from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt


# ============================================================
# Paths
# ============================================================

DATA_DIR = Path(
    "/content/drive/MyDrive/SISTER/data"
)

MODEL_DIR = DATA_DIR / "models"
PREDICTION_DIR = DATA_DIR / "predictions"

OUTPUT_DIR = DATA_DIR / "visualizations"


# ============================================================
# Load prediction results
# ============================================================

def load_prediction_data():
    """
    Load predictions and corresponding test targets.
    """

    prediction_path = (
        PREDICTION_DIR / "predicted_lst.npy"
    )

    target_path = (
        MODEL_DIR / "test_targets.npy"
    )

    if not prediction_path.exists():

        raise FileNotFoundError(
            f"Predictions not found:\n"
            f"{prediction_path}"
        )

    if not target_path.exists():

        raise FileNotFoundError(
            f"Test targets not found:\n"
            f"{target_path}"
        )

    predictions = np.load(
        prediction_path
    )

    targets = np.load(
        target_path
    )

    return predictions, targets


# ============================================================
# Prepare single patch
# ============================================================

def get_patch(
    array,
    index=0
):
    """
    Extract one LST patch.

    Input shape:

        (N, 1, H, W)

    Output shape:

        (H, W)
    """

    if index < 0 or index >= len(array):

        raise IndexError(
            f"Patch index {index} is out of range."
        )

    return array[
        index,
        0
    ]


# ============================================================
# Plot reference and prediction
# ============================================================

def plot_lst_comparison(
    prediction,
    target,
    index=0
):
    """
    Display reference LST, predicted LST,
    and prediction error.
    """

    error = (
        prediction
        -
        target
    )

    figure, axes = plt.subplots(
        1,
        3,
        figsize=(18, 6)
    )


    # --------------------------------------------------------
    # Reference LST
    # --------------------------------------------------------

    image_1 = axes[0].imshow(
        target,
        cmap="inferno"
    )

    axes[0].set_title(
        "Reference LST"
    )

    axes[0].set_xlabel(
        "Pixel"
    )

    axes[0].set_ylabel(
        "Pixel"
    )

    figure.colorbar(
        image_1,
        ax=axes[0],
        label="LST"
    )


    # --------------------------------------------------------
    # Predicted LST
    # --------------------------------------------------------

    image_2 = axes[1].imshow(
        prediction,
        cmap="inferno"
    )

    axes[1].set_title(
        "Predicted LST"
    )

    axes[1].set_xlabel(
        "Pixel"
    )

    axes[1].set_ylabel(
        "Pixel"
    )

    figure.colorbar(
        image_2,
        ax=axes[1],
        label="LST"
    )


    # --------------------------------------------------------
    # Error
    # --------------------------------------------------------

    image_3 = axes[2].imshow(
        error,
        cmap="coolwarm"
    )

    axes[2].set_title(
        "Prediction Error"
    )

    axes[2].set_xlabel(
        "Pixel"
    )

    axes[2].set_ylabel(
        "Pixel"
    )

    figure.colorbar(
        image_3,
        ax=axes[2],
        label="Prediction - Reference"
    )


    figure.suptitle(
        f"LST Prediction Comparison - Patch {index}"
    )

    figure.tight_layout()


    return figure


# ============================================================
# Save comparison figure
# ============================================================

def save_lst_comparison(
    prediction,
    target,
    index=0
):
    """
    Save the LST comparison figure.
    """

    OUTPUT_DIR.mkdir(
        parents=True,
        exist_ok=True
    )

    figure = plot_lst_comparison(
        prediction,
        target,
        index
    )

    output_path = (
        OUTPUT_DIR
        /
        f"lst_comparison_patch_{index}.png"
    )

    figure.savefig(
        output_path,
        dpi=300,
        bbox_inches="tight"
    )

    plt.close(
        figure
    )

    print(
        "Saved:",
        output_path
    )

    return output_path


# ============================================================
# Plot training history
# ============================================================

def plot_training_history():
    """
    Plot training and validation losses.
    """

    history_path = (
        MODEL_DIR
        /
        "training_history.npy"
    )

    if not history_path.exists():

        raise FileNotFoundError(
            f"Training history not found:\n"
            f"{history_path}"
        )

    history = np.load(
        history_path,
        allow_pickle=True
    ).item()


    epochs = range(
        1,
        len(history["train_loss"]) + 1
    )


    figure, axes = plt.subplots(
        1,
        3,
        figsize=(18, 5)
    )


    # --------------------------------------------------------
    # Total loss
    # --------------------------------------------------------

    axes[0].plot(
        epochs,
        history["train_loss"],
        label="Training"
    )

    axes[0].plot(
        epochs,
        history["validation_loss"],
        label="Validation"
    )

    axes[0].set_title(
        "Total Loss"
    )

    axes[0].set_xlabel(
        "Epoch"
    )

    axes[0].set_ylabel(
        "Loss"
    )

    axes[0].legend()


    # --------------------------------------------------------
    # Data loss
    # --------------------------------------------------------

    axes[1].plot(
        epochs,
        history["train_data_loss"],
        label="Training"
    )

    axes[1].plot(
        epochs,
        history["validation_data_loss"],
        label="Validation"
    )

    axes[1].set_title(
        "Data Loss"
    )

    axes[1].set_xlabel(
        "Epoch"
    )

    axes[1].set_ylabel(
        "MSE"
    )

    axes[1].legend()


    # --------------------------------------------------------
    # Physics loss
    # --------------------------------------------------------

    axes[2].plot(
        epochs,
        history["train_physics_loss"],
        label="Training"
    )

    axes[2].plot(
        epochs,
        history["validation_physics_loss"],
        label="Validation"
    )

    axes[2].set_title(
        "Physics Loss"
    )

    axes[2].set_xlabel(
        "Epoch"
    )

    axes[2].set_ylabel(
        "Physics Loss"
    )

    axes[2].legend()


    figure.tight_layout()


    OUTPUT_DIR.mkdir(
        parents=True,
        exist_ok=True
    )


    output_path = (
        OUTPUT_DIR
        /
        "training_history.png"
    )


    figure.savefig(
        output_path,
        dpi=300,
        bbox_inches="tight"
    )

    plt.close(
        figure
    )


    print(
        "Saved:",
        output_path
    )

    return output_path


# ============================================================
# Run all visualizations
# ============================================================

def create_visualizations():

    print(
        "Creating visualizations..."
    )


    # --------------------------------------------------------
    # Load predictions and targets
    # --------------------------------------------------------

    predictions, targets = (
        load_prediction_data()
    )


    if predictions.shape != targets.shape:

        raise ValueError(
            "Predictions and targets must have "
            "the same shape."
        )


    print(
        "Prediction shape:",
        predictions.shape
    )

    print(
        "Target shape:",
        targets.shape
    )


    # --------------------------------------------------------
    # Use first test patch
    # --------------------------------------------------------

    prediction = get_patch(
        predictions,
        index=0
    )

    target = get_patch(
        targets,
        index=0
    )


    # --------------------------------------------------------
    # Save comparison
    # --------------------------------------------------------

    save_lst_comparison(
        prediction,
        target,
        index=0
    )


    # --------------------------------------------------------
    # Save training curves
    # --------------------------------------------------------

    plot_training_history()


# ============================================================
# Main
# ============================================================

if __name__ == "__main__":

    create_visualizations()